# Smart Semsar — furniture pipeline on Colab (`inference.py`)

تجربة كل خطوات الفرش على إعلان واحد: المخطط → العفش → المراجعة → جولة 3D، وكمان قطعة فرش من صورتك (TRELLIS.2).

**What it does:** runs `inference.py` for one listing and shows the furnished 3D walkthrough right here.

| Run | What you see | Needs |
|---|---|---|
| A | the listing furnished by the rules | nothing |
| B | the same with your wishes (density, must-have, exclude) | nothing |
| C | **your own piece from a photo**, made 3D by TRELLIS.2 and placed in the room | `HF_TOKEN` |
| D | furniture chosen and arranged by Gemini | `GEMINI_API_KEY` |

**No GPU needed** — keep the default CPU runtime. TRELLIS.2 runs on Hugging Face's GPU (free daily quota).

**Before you run** — open the key icon (Secrets) on the left, add these with *Notebook access* on:
- `HF_TOKEN` — a Hugging Face **Read** token (https://huggingface.co/settings/tokens) — for run C
- `GEMINI_API_KEY` — optional, free at https://aistudio.google.com/apikey — reads the photo and runs D
- `GITHUB_TOKEN` — only if the repository is private (a token with read access to it)

Then `Runtime → Run all`.

## 1. Settings

In [ ]:
REPO = "ghadeersalah42/SmartSemsar"
BRANCH = "furniture-merge"      # the branch with the merged furniture work
PROPERTY_ID = "PROP_1002"       # any property_id from data/final_merged_dataset.csv

## 2. Get the code
Clones the branch from GitHub. If the branch is not on GitHub yet, upload `furniture-merge.bundle`
when asked (it is applied on top of `3D-Plan`).

In [ ]:
import os, subprocess
from google.colab import userdata

def secret(name):
    try:
        return userdata.get(name) or ""
    except Exception:          # secret missing or notebook access off
        return ""

gh = secret("GITHUB_TOKEN")
url = f"https://{gh}@github.com/{REPO}.git" if gh else f"https://github.com/{REPO}.git"
os.chdir("/content")
if not os.path.isdir("SmartSemsar"):
    ok = subprocess.run(["git", "clone", "-q", "--depth", "50", "-b", BRANCH, url, "SmartSemsar"]).returncode == 0
    if not ok:
        print(f"Branch {BRANCH} not found on GitHub: cloning 3D-Plan and applying the bundle instead.")
        subprocess.run(["git", "clone", "-q", "-b", "3D-Plan", url, "SmartSemsar"], check=True)
        from google.colab import files
        print("Upload furniture-merge.bundle:")
        name = next(iter(files.upload()))
        subprocess.run(["git", "-C", "SmartSemsar", "fetch", "-q", f"/content/{name}", f"{BRANCH}:{BRANCH}"], check=True)
        subprocess.run(["git", "-C", "SmartSemsar", "checkout", "-q", BRANCH], check=True)
os.chdir("/content/SmartSemsar")
print(subprocess.run(["git", "log", "--oneline", "-3"], capture_output=True, text=True).stdout)

## 3. Install (about a minute)

In [ ]:
!pip install -q shapely trimesh scipy mapbox_earcut pandas pydantic pillow requests gradio_client

## 4. Keys
Read from Colab Secrets into the environment; only whether each one is set is printed.

In [ ]:
for name in ("HF_TOKEN", "GEMINI_API_KEY", "GROQ_API_KEY"):
    value = secret(name)
    if value:
        os.environ[name] = value
    print(f"{name:15s} {'set' if value else 'not set'}")

## 5. Viewer
Serves the `out/` folder on a local port and shows a walkthrough page inside the notebook.
In the viewer: drag to look around, **Walk** + W A S D to move, or open it full screen with the link.

In [ ]:
import subprocess, time
from google.colab import output

server = subprocess.Popen(["python", "-m", "http.server", "8000", "--directory", "out"],
                          stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
os.makedirs("out", exist_ok=True)
time.sleep(1)

def show(property_id=PROPERTY_ID, height=720):
    page = f"/{property_id}/walkthrough.html"
    if not os.path.exists("out" + page):
        print("No walkthrough built for", property_id, "- see the messages above.")
        return
    output.serve_kernel_port_as_iframe(8000, path=page, height=height)

## Run A — the listing furnished by the rules

In [ ]:
!python inference.py $PROPERTY_ID --no-open
show()

## Run B — with your wishes
`--density minimal|normal|full`, `--must-have` / `--exclude` catalog names (sofa, desk, tv, armchair ...),
`--dining-seats 4|6`, `--style-brief` free text.

In [ ]:
!python inference.py $PROPERTY_ID --density full --must-have desk armchair --exclude tv --dining-seats 4 --no-open
show()

## Run C — your own furniture from a photo (TRELLIS.2)
Upload a photo of **one** piece, fully visible (a plain background helps).
Set `KIND` (sofa, armchair, bed, desk, ...) and its real `WIDTH_M`; with `GEMINI_API_KEY` set you can leave
`KIND = ""` and the photo is recognised. Takes about 40 s; uses your Hugging Face GPU quota.

In [ ]:
from google.colab import files

KIND = "armchair"     # "" = let Gemini read it from the photo
WIDTH_M = 0.8         # real width in meters; 0 = the catalog piece's width

uploaded = files.upload()
PHOTO = "/content/" + next(iter(uploaded))
with open(PHOTO, "wb") as f:
    f.write(uploaded[next(iter(uploaded))])
opts = (f"--kind {KIND} " if KIND else "") + (f"--width {WIDTH_M} " if WIDTH_M else "")
!python inference.py $PROPERTY_ID --photo "$PHOTO" $opts --generator trellis --no-open
show()

## Run D (optional) — Gemini chooses and arranges the furniture
The model writes placement instructions ("sofa on wall B"); the code computes the positions and the
validator checks them. Without `GEMINI_API_KEY` this falls back to the rules and says so.

In [ ]:
STYLE_BRIEF = "warm and bright, light wood, a reading corner in the living room"
!python inference.py $PROPERTY_ID --planner llm --style-brief "$STYLE_BRIEF" --no-open
show()

## Download the results
`staging.json`, `walkthrough.html` (opens offline in any browser) and your custom model.

In [ ]:
import shutil
shutil.make_archive(f"/content/{PROPERTY_ID}_results", "zip", f"out/{PROPERTY_ID}")
files.download(f"/content/{PROPERTY_ID}_results.zip")